## Stage 0: raw data -> SpatialData

# MERFISH mouse ileum: raw data -> SpatialData -> precomputed -> view

This notebook inlines the **real, actual code** from the repo's own 4-stage example (`examples/merfish_mouse_ileum/0_raw_to_spatialdata.py` through `3_precomputed_to_spatialdata.py`), split along the scripts' own `##` cell-marker convention.

**Known, real caveats:**
- Dataset is **2.5D** -- only 9 discrete Z layers, not a dense stack.
- Raw channel axis order is verified by construction; **Cellpose label** axis order is not (relies on SpatialData's default guess).
- Real physical Z-calibration (`z_um`) was never completed upstream.
- Stage 1's raster/mesh conversion is commented out by default -- only points/annotations run.
- Stage 3's mesh-reading is unfinished upstream (commented out, TODO).
- DAPI and membrane labels use different index values for the same physical cell (noted in the original code).

Run `pip install -e .` (or `uv sync`) first. All data lives under `data/merfish_mouse_ileum/` (`raw/` for downloaded/input data, `out/` for everything generated).

In [ ]:
%load_ext jupyter_black

In [1]:
from pathlib import Path

dataset_path = Path.cwd().parent.parent / "data" / "merfish_mouse_ileum"
raw_path = dataset_path / "raw"
out_path = dataset_path / "out"
raw_path.mkdir(parents=True, exist_ok=True)
out_path.mkdir(parents=True, exist_ok=True)

Imports, download the dataset, parse raw DAPI/membrane channels and transcript coordinates, build the initial SpatialData object.

In [ ]:
import numpy as np
from pathlib import Path
import subprocess
import hashlib
import spatialdata as sd
from dask_image.imread import imread
import pandas as pd
from anndata import AnnData
import geopandas as gpd
from shapely import Point, Polygon
from scipy.optimize import curve_fit
import dask.array as da
from spatialdata.models import points_dask_dataframe_to_geopandas
from geopandas import sjoin
from geopandas import GeoDataFrame

pd.set_option("future.infer_string", False)
# download the data: https://datadryad.org/dataset/doi:10.5061/dryad.jm63xsjb2
download_path = raw_path / "data_release_baysor_merfish_gut.zip"
unzipped_path = raw_path / "data_release_baysor_merfish_gut"

# download the example data
CHECKSUM_DOWNLOAD = "501a206666b5895e9182245dda8d4e60"
#
if (
    not download_path.exists()
    or CHECKSUM_DOWNLOAD != hashlib.md5(download_path.read_bytes()).hexdigest()
):
    print(
        "Data missing or wrong checksum. Please download the data from "
        "https://datadryad.org/dataset/doi:10.5061/dryad.jm63xsjb2"
    )

# unzip the downloaded file
if not unzipped_path.exists():
    subprocess.run(
        f'unzip -o "{download_path}" -d "{raw_path}"', shell=True, check=True
    )

# parse raw images

dapi_data = imread(unzipped_path / "raw_data" / "dapi_stack.tif")
dapi_data = da.reshape(dapi_data, (1, *dapi_data.shape))

membrane_data = imread(unzipped_path / "raw_data" / "membrane_stack.tif")
membrane_data = da.reshape(membrane_data, (1, *membrane_data.shape))

# the data is in the format (channel, z, y, x)
data = da.concatenate([dapi_data, membrane_data], axis=0)
img_stack = sd.models.Image3DModel.parse(
    data, scale_factors=[2, 2], c_coords=["DAPI", "Membrane"]
)

# parse transcripts locations
points_path = unzipped_path / "raw_data" / "molecules.csv"
df = pd.read_csv(points_path)
molecules = sd.models.PointsModel.parse(
    df,
    coordinates={"x": "x_pixel", "y": "y_pixel", "z": "z_pixel"},
    feature_key="gene",
)


def affine(x, a, b):
    return a * x + b


# infer the pixel size data for the image data using the molecules data
# this is explained in the README.txt (and can and easily seen from the data)
def z_raw_to_layer_index(z_raw: float) -> float:
    return (z_raw - 2.5) / 1.5 + 1


def layer_index_to_z_raw(layer_index: float) -> float:
    return (layer_index - 1) * 1.5 + 2.5


# quick sanity-check
assert 3 == z_raw_to_layer_index(layer_index_to_z_raw(3))

z_pixels_values = df.z_pixel.value_counts().sort_index().index.tolist()
(a, b), _ = curve_fit(affine, z_pixels_values, list(range(1, 10)))

df = df.assign(layer=lambda df: affine(x=df["z_pixel"], a=a, b=b))
df["layer"] = df["layer"].round(0).astype(int)

affine_correct_z_pixel_raster = sd.transformations.Affine(
    [
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1 / a, -b / a],
        [0, 0, 0, 1],
    ],
    input_axes=("x", "y", "z"),
    output_axes=("x", "y", "z"),
)
# (a_z, b_z), _ = curve_fit(affine, df.z_pixel, df.z_um)
# (a_y, b_y), _ = curve_fit(affine, df.y_pixel, df.y_um)
# (a_x, b_x), _ = curve_fit(affine, df.x_pixel, df.x_um)
#
# pixel_to_um = sd.transformations.Affine(
#     [[a_x, 0, 0, b_x], [0, a_y, 0, b_y], [0, 0, a_z, b_z], [0, 0, 0, 1]],
#     input_axes=("x", "y", "z"),
#     output_axes=("x", "y", "z"),
# )
# # to be more accurate we could reconstruct the matrix from scratch as above
# um_to_pixel = pixel_to_um.inverse()

sd.transformations.set_transformation(
    img_stack,
    transformation=affine_correct_z_pixel_raster,
    to_coordinate_system="global",
)
sdata = sd.SpatialData.init_from_elements(
    {
        "stains": img_stack,
        "molecules": molecules,
    }
)

# parse cellpose segmentation (cell centroids, counts, cluster assignment)
df_coords = pd.read_csv(
    unzipped_path / "data_analysis/cellpose/segmentation/cell_coords.csv"
)
df_counts = pd.read_csv(
    unzipped_path / "data_analysis/cellpose/segmentation/segmentation_counts.tsv",
    sep="\t",
)
df_cluster = pd.read_csv(
    unzipped_path / "data_analysis/cellpose/clustering/cell_assignment.csv",
)

x = df_counts.iloc[:, range(1, df_counts.shape[1])].values.T
cell_ids = np.arange(1, x.shape[0] + 1)
assert np.array_equal(df_cluster["cell"], cell_ids)
var_name = df_counts.iloc[:, 0]
obs = pd.DataFrame({"cluster": df_cluster["leiden_final"]})
adata = AnnData(X=x, var=pd.DataFrame(index=var_name), obs=obs)
adata.obs["region"] = "cells_centroids_cellpose"
adata.obs["region"] = adata.obs["region"].astype("category")
adata.obs["cell_id"] = cell_ids
adata = sd.models.TableModel.parse(
    adata,
    region="cells_centroids_cellpose",
    region_key="region",
    instance_key="cell_id",
)
df_coords.index = cell_ids
cells = sd.models.PointsModel.parse(df_coords)

sdata["cells_centroids_cellpose"] = cells
sdata["gene_expression_cellpose"] = adata

# parse cellpose segmentation (dapi and membrane labels)
data = imread(
    unzipped_path / "data_analysis/cellpose/cell_boundaries/results/cellpose_dapi.tif"
)
dapi_labels = sd.models.Labels3DModel.parse(
    data,
    transformations={"global": affine_correct_z_pixel_raster},
    scale_factors=[2, 2],
)
data = imread(
    unzipped_path
    / "data_analysis/cellpose/cell_boundaries/results/cellpose_membrane.tif"
)
membrane_labels = sd.models.Labels3DModel.parse(
    data,
    transformations={"global": affine_correct_z_pixel_raster},
    scale_factors=[2, 2],
)
# problem in the data: the same cell across dapi_labels and membrane_labels have
# different index value
sdata["dapi_labels"] = dapi_labels
sdata["membrane_labels"] = membrane_labels

Parse the Baysor segmentation output (2.5D shapes, per-cell stats, gene counts) into a Table.

In [ ]:
# parse baysor segmentation (2.5D shapes, segmentation cell stats, counts"
df_segmentation = pd.read_csv(
    unzipped_path / "data_analysis/baysor/segmentation/segmentation.csv"
)
df_cell_stats = pd.read_csv(
    unzipped_path / "data_analysis/baysor/segmentation/segmentation_cell_stats.csv"
)
df_counts = pd.read_csv(
    unzipped_path / "data_analysis/baysor/segmentation/segmentation_counts.tsv",
    sep="\t",
)

x = df_counts.iloc[:, range(1, df_counts.shape[1])].values.T
cell_ids = np.arange(1, x.shape[0] + 1)
assert np.array_equal(df_cell_stats["cell"], cell_ids)

adata = AnnData(
    X=x,
    var=pd.DataFrame(index=df_counts.iloc[:, 0]),
    obs=pd.DataFrame({"cell_id": cell_ids, "region": "cells_circles_baysor"}),
)
adata.obs["region"] = adata.obs["region"].astype("category")
adata = sd.models.TableModel.parse(
    adata, region="cells_circles_baysor", region_key="region", instance_key="cell_id"
)

Merge cell stats into the table's `obs` (axis=1 bug fixed).

In [ ]:
adata.obs = pd.merge(
    adata.obs,
    df_cell_stats.drop(columns=["x", "y"]),
    left_on="cell_id",
    right_on="cell",
    how="left",
).drop(columns=["cell"])

Build circular cell shapes from centroid + area; store expression/molecule tables.

In [ ]:
xy = df_cell_stats[["x", "y"]].values
radii = (df_cell_stats["area"].to_numpy() / np.pi) ** 0.5
gdf = gpd.GeoDataFrame(
    {"radius": radii},
    geometry=gpd.GeoSeries([Point(xy[i, 0], xy[i, 1]) for i in range(len(xy))]),
    index=df_cell_stats["cell"],
)
print(
    "Baysor segmentation: {}/{} cells have NaN area; dropping them".format(
        np.sum(df_cell_stats["area"].isna()), len(df_cell_stats)
    )
)
gdf = gdf[~gdf.radius.isna()]
gdf = sd.models.ShapesModel.parse(gdf)

# note, the transcripts from baysor have the same coordinates and order as the
# raw transcripts, but since we have 2 different baysor segmentations, we keep all of
# them in separate objects
assert np.array_equal(molecules["x"].compute(), df_segmentation["x"])
assert np.array_equal(molecules["y"].compute(), df_segmentation["y"])
assert np.allclose(molecules["z"].compute(), df_segmentation["z"])

points = sd.models.PointsModel.parse(df_segmentation, feature_key="gene")
points["cell"] = points["cell"].round(0).astype(int)

sdata["gene_expression_baysor"] = adata
sdata["cells_circles_baysor"] = gdf
sdata["molecule_baysor"] = points

# TODO: here parse "poly_per_z.json"

# we could also parse "baysor_membrane_prior". It is analogous to the above except that
# "poly_per_z.json" is missing

Load the per-Z-layer polygon file and parse its custom JSON into real polygons.

In [ ]:
path = unzipped_path / "data_analysis/baysor/segmentation/poly_per_z.json"
# the poly_per_z.json file seems to be using a legacy format and it's not geojson, see
# more here: https://github.com/kharchenkolab/Baysor/issues/129
# newer versions of Baysor use GeoJSON:
# https://github.com/kharchenkolab/Baysor/blob/master/CHANGELOG.md#071--2024-11-19
# this works for GeoJSON files:
# polygons = gpd.read_file("GeoJSON:" + str(path))


# let's parse the JSON file manually
def geometry_from_dict(d):
    geom_type = d.get("type")
    coords = d.get("coordinates")
    if geom_type == "Polygon":
        return Polygon(coords[0])
    # untested:
    # elif geom_type == 'MultiPolygon':
    #     return MultiPolygon([Polygon(p[0]) for p in coords])
    else:
        raise ValueError(f"Unsupported geometry type: {geom_type}")


df = pd.read_json(path)

Convert each Z-layer's polygons into a GeoDataFrame.

In [ ]:
shapes_per_layer = {}
skipped = 0
for row in df.itertuples():
    # print(row._fields)  # gives ('Index', 'z_id', 'geometries', 'type')
    gdf = gpd.GeoDataFrame(
        geometry=gpd.GeoSeries([geometry_from_dict(shape) for shape in row.geometries]),
    )
    gdf["geometry"] = gdf["geometry"].apply(lambda geom: geom.buffer(0))
    gdf["layer"] = row.z_id
    # some geometries are empty (I haven't checked if this happens in the raw data or
    # after calling .buffer(0))
    mask = gdf.geometry.apply(lambda geom: not geom.is_empty)
    gdf = gdf[mask]
    gdf = sd.models.ShapesModel.parse(gdf)
    shapes_per_layer[row.z_id] = gdf
    # TODO: verify that it's not needed (old way to fix scale factor z)
    # z_raw = layer_index_to_z_raw(row.z_id)
    # gdf["z"] = (z_raw / a).item()
    print(f"Layer {row.z_id}: {len(gdf)} shapes, skipped {skipped} empty geometries")

Store each Z-layer's shapes into the SpatialData object.

In [ ]:
for layer, shapes in shapes_per_layer.items():
    sdata[f"cells_layer_{layer}_baysor"] = shapes

Assign each transcript to a Z-layer; match each polygon to its most common overlapping cell ID.

In [ ]:
# 1-based indexing
z_layer = (
    points["z_raw"]
    .apply(z_raw_to_layer_index, meta=("z_raw", "float64"))
    .astype(int)
    .compute()
)
points["layer"] = z_layer

for layer in list(shapes_per_layer.keys()):
    shapes_in_layer = shapes_per_layer[layer]
    points_in_layer = points[points["layer"] == layer]
    # sjoin to transfer the column ('cell') from the points to the shapes

    points_geopandas = points_dask_dataframe_to_geopandas(points_in_layer)

    joined = sjoin(
        left_df=shapes_in_layer,
        right_df=points_geopandas,
        how="left",
        predicate="contains",
    )["cell"]
    # cells with no points gets set to 0 (background)
    joined = joined.fillna(0).astype(int)
    df = pd.DataFrame({"shape": joined.index, "assigned_cell": joined.tolist()})
    # group by 'shape' and get the most frequent 'assigned_cell' for each shape
    most_abundant = (
        df.groupby("shape")["assigned_cell"]
        # get the first value in case of a tie
        .agg(lambda x: x.mode()[0])
        .reset_index()
        .rename(columns={"assigned_cell": "most_abundant_cell"})
    )
    most_abundant.set_index("shape", inplace=True)
    shapes_in_layer["most_abundant_cell"] = most_abundant["most_abundant_cell"]
    # ##
    # # debug
    # some asserts in the for loop will fail! this because for some reasons points
    # that are at the border of shapes are selected by "contains" in sjoin, but fail
    # the .contains() check below. This is fine.
    # for shape_iloc in range(1000):
    #     cell = shapes_in_layer.iloc[shape_iloc].most_abundant_cell.item()
    #     if cell == 0:
    #         print(f"Skipping shape (iloc) {shape_iloc} with cell {cell}")
    #     else:
    #         point_iloc = np.where(
    #             points_geopandas.cell == cell
    #         )[0][0]
    #         print(point_iloc)
    #         points_geopandas.iloc[point_iloc]
    #         assert shapes_in_layer.iloc[shape_iloc].geometry.contains(
    #             points_geopandas.iloc[point_iloc].geometry
    #         )
    #         polygon = shapes_in_layer.iloc[shape_iloc].geometry
    #         point = points_geopandas.iloc[point_iloc].geometry
    #         import matplotlib.pyplot as plt
    #         x, y = polygon.exterior.xy
    #         plt.plot(x, y, 'b-', linewidth=2, label='Polygon')
    #         plt.fill(x, y, color='blue', alpha=0.2)
    #         plt.plot(point.x, point.y, 'ro', label='Point')
    #         plt.show()
    #
    # ##
    pass

# shapes_per_layer[2]
# merge the shapes_per_layer into a single GeoDataFrame
gdf_all_layers = pd.concat(shapes_per_layer.values(), ignore_index=True)
gdf_all_layers = gpd.GeoDataFrame(gdf_all_layers, geometry="geometry")

Helper functions (from a hackathon contribution) for matching polygons across adjacent Z-layers.

In [ ]:
# code from owkin hackathon from Karen Herreman and Quentin Blampey
def match_cells_iomin(
    layer1: GeoDataFrame, layer2: GeoDataFrame, threshold: float
) -> dict[int, list[int]]:
    """
    Matches polygons between two layers based on Intersection over Minimum Area (IoMin).

    Parameters
    ----------
    layer1
        The first set of polygons.
    layer2
        The second set of polygons.
    threshold
        IoMin threshold for considering polygons as matching.

    Returns
    -------
    dict[int, list[int]]
        Dictionary where keys are indices from `layer1` and values are lists of matching indices from `layer2`.
    """
    matched_indices: dict[int, list[int]] = {}
    indices_layer1, indices_layer2 = list(
        layer2.sindex.query(layer1["geometry"], predicate="intersects")
    )

    # Map the positional indices to label-based indices
    label_indices_layer1 = layer1.index[indices_layer1]
    label_indices_layer2 = layer2.index[indices_layer2]

    for i in range(len(label_indices_layer1)):
        idx1 = label_indices_layer1[i]
        idx2 = label_indices_layer2[i]
        poly1 = layer1.loc[idx1].geometry
        poly2 = layer2.loc[idx2].geometry

        # Compute overlap area and mean area
        overlap_area = poly1.intersection(poly2).area
        min_area = min(poly1.area, poly2.area)

        # Calculate IoMean and apply threshold
        iomax = overlap_area / min_area
        if iomax >= threshold:
            matched_indices.setdefault(idx1, []).append(idx2)

    return matched_indices


# process_pseudo3D_shapes() and match_cells_iomin() are contributed from Karen Herreman
# and Quentin Blampey during the scverse <> owkin hackathon
def process_pseudo3D_shapes(
    gdf: GeoDataFrame, threshold: float = 0.3, cell_column: str = "cell_id"
) -> GeoDataFrame:
    """
    Combines layers in a GeoDataFrame, assigning unique cell indices based on
    spatial matches between polygons in adjacent layers.

    Parameters
    ----------
    gdf
        GeoDataFrame with layers to process.
    threshold
        IoMean threshold for matching polygons between layers.
    cell_column
        Name of the column in the updated GeoDataFrame containing the unique cell indices.

    Returns
    -------
    GeoDataFrame
        Updated GeoDataFrame with the `cell_column` column populated.
    """
    nb_layers = sorted(gdf["layer"].unique())
    cell_index = 0
    if cell_column in gdf.columns:
        raise ValueError(
            f"Column '{cell_column}' already exists in the GeoDataFrame. "
            "Please choose a different value for `cell_column`."
        )
    gdf[cell_column] = None

    # Process each pair of adjacent layers
    for a, b in zip(nb_layers[:-1], nb_layers[1:]):
        layer_a = gdf[gdf["layer"] == a]
        layer_b = gdf[gdf["layer"] == b]

        # Match polygons between the layers
        matched_indices = match_cells_iomin(layer_a, layer_b, threshold)

        # Update cell indices based on matches
        for idx1, idx2_list in matched_indices.items():
            for idx2 in idx2_list:
                if gdf.loc[idx1, cell_column] is not None:
                    gdf.loc[idx2, cell_column] = gdf.loc[idx1, cell_column]
                else:
                    gdf.loc[idx1, cell_column] = cell_index
                    gdf.loc[idx2, cell_column] = cell_index
                    cell_index += 1

    unmatched = gdf[cell_column].isna()
    gdf.loc[unmatched, cell_column] = range(cell_index, cell_index + unmatched.sum())

    return gdf

Stitch per-layer 2D polygons into pseudo-3D cells; write the completed SpatialData object to disk.

In [ ]:
import pandas as pd

print(pd.get_option("future.infer_string"))

In [ ]:
params, _ = curve_fit(affine, points.z_raw.compute(), points.z.compute())
a, b = params

gdf_all_layers = gdf_all_layers.assign(
    z_raw=lambda df: df["layer"].apply(layer_index_to_z_raw)
).assign(z=lambda df: affine(x=df["z_raw"], a=a, b=b))


gdf_all_layers = sd.models.ShapesModel.parse(gdf_all_layers)

processed = process_pseudo3D_shapes(
    gdf_all_layers, threshold=0.3, cell_column="cell_id"
)

# sdata["cells_baysor"] = gdf_all_layers
sdata["cells_baysor"] = processed

sdata.write(out_path / "merfish_mouse_ileum.sdata.zarr", overwrite=True)


# ##
# Interactive(sdata)

## Stage 1: SpatialData -> precomputed

Imports and output paths.

In [ ]:
import napari_spatialdata.constants.config
import spatialdata as sd
from pathlib import Path
from numpy.random import default_rng

from tissue_map_tools.igneous_converters import (  # noqa: F401
    from_spatialdata_raster_to_sharded_precomputed_raster_and_meshes,
)
from tissue_map_tools.data_model.annotations_utils import (
    make_dtypes_compatible_with_precomputed_annotations,
)
import time  # noqa: F401
import shutil  # noqa: F401
from tissue_map_tools.converters import (  # noqa: F401
    from_spatialdata_points_to_precomputed_points,
)

RNG = default_rng(42)

napari_spatialdata.constants.config.PROJECT_3D_POINTS_TO_2D = False
napari_spatialdata.constants.config.PROJECT_2_5D_SHAPES_TO_2D = False

sdata_zarr_path = out_path / "merfish_mouse_ileum.sdata.zarr"
precomputed_path = out_path / "merfish_mouse_ileum_precomputed"

Load the SpatialData object written by Stage 0.

In [ ]:
# load the data
f = Path(sdata_zarr_path)
sdata = sd.read_zarr(f)
# print(sd.get_extent(sdata["molecules"]))

Crop to a small bounding box for a manageable example region.

In [ ]:
# subset the data
sdata_small = sd.bounding_box_query(
    sdata,
    axes=("x", "y", "z"),
    min_coordinate=[4000, 0, -10],
    max_coordinate=[5000, 1500, 200],
    target_coordinate_system="global",
)

(Unused alternative crop, left as reference.)

Select and clean transcript columns for precomputed-annotations conversion.

In [ ]:
subset = RNG.choice(len(sdata["molecule_baysor"]), 100, replace=False)

print(sdata["molecule_baysor"].columns)
# subset_df = sdata["molecule_baysor"].compute().iloc[subset]
subset_df = sdata["molecule_baysor"].compute()
subset_df = subset_df[["x", "y", "z", "gene"]]

sdata["molecule_baysor"] = sd.models.PointsModel.parse(
    make_dtypes_compatible_with_precomputed_annotations(
        subset_df,
        max_categories=250,
        check_for_overflow=True,
    )
)

# TODO: temporary workaround: raster data converted to precomputed expresses units in nm
#  therefore let's multiply the points by 1000
for ax in ["x", "y", "z"]:
    sdata["molecule_baysor"][ax] = sdata["molecule_baysor"][ax] * 1000 + RNG.random()

Debug print of two example points, to sanity-check dtypes.

In [ ]:
# debug
points = sdata["molecule_baysor"].compute().iloc[:2]
print("point 0")
print(points.iloc[0])
print("")
print("point 1")
print(points.iloc[1])
print("")
print(points.x.dtype)
# print(points.gene.cat.categories)
print(points.gene.cat.categories.get_loc(points.gene.iloc[0]))

Convert the transcript points to precomputed annotations format.

In [ ]:
print("converting the points to the precomputed format")

# TODO: there should be no need to add the subpath (we should be able to specify the
#  parent cloud volume object
# TODO: the info file in the parent volume should be updated to include the points
# TODO: the view APIs show include the points

start = time.time()
path = precomputed_path / "molecule_baysor"
if path.exists():
    shutil.rmtree(path)
from_spatialdata_points_to_precomputed_points(
    sdata["molecule_baysor"],
    precomputed_path=precomputed_path,
    points_name="molecule_baysor",
    limit=10000,
    # limit=500,
    sharded=True,
)
print(f"conversion of points: {time.time() - start}")

## Stage 2: Visualize the precomputed output in Neuroglancer

In [ ]:
from tissue_map_tools.view import (  # noqa: F401
    view_precomputed_in_napari,
    view_precomputed_in_neuroglancer,
    view_precomputed_in_vitessce,
    compute_initial_camera_state,
)
from pathlib import Path

precomputed_path = out_path / "merfish_mouse_ileum_precomputed"


viewer = view_precomputed_in_neuroglancer(
    data_path=str(precomputed_path),
)
# TODO - napari did not open it
# view_precomputed_in_napari(
#     data_path= str(precomputed_path),
#     show_meshes=False,
#     show_raster=True,
#     show_points=True,
# )

initial_camera_state = compute_initial_camera_state(
    data_path=str(precomputed_path),
)

annotation_options = {
    "projectionAnnotationSpacing": 2.45,
    "featureIndexProp": "gene",
}

viewer = view_precomputed_in_vitessce(
    data_path=str(precomputed_path),
    initial_camera_state=initial_camera_state,
    use_web_app=True,
    show_annotations=True,
    show_meshes=True,
    obs_type_annotation="point",
    obs_type_segmentation="cell",
    annotation_options=annotation_options,
    obsColorEncoding="cellSetSelection",
)
viewer

In [2]:
from tissue_map_tools.vitessce_configs.layer_specs import SegmentationLayerSpec, AnnotationLayerSpec, SpatialDataObsSpec
from tissue_map_tools.vitessce_configs.neuroglancer_config_builder import build_neuroglancer_config
precomputed_mesh_path = str(out_path / "merfish_mouse_ileum_precomputed")
sdata_path = str (out_path / "merfish_mouse_ileum.sdata.zarr")
precomputed_annotations_path = str (out_path / "merfish_mouse_ileum_precomputed_annotations/molecule_baysor")

print(precomputed_annotations_path)
vc = build_neuroglancer_config(
    name="MERFISH mouse ileum dataset",
    schema_version="1.0.18",
    segmentations=[
        SegmentationLayerSpec(
            file_uid="merfish-meshes", 
            local_path=precomputed_mesh_path,
            obs_type="cell", 
            label="Meshes",
            feature_type="",
            auto_generate_obs_sets=False,   # real Cell Types sets come from spatialdata_obs below
            options={"subsources": {"default": True, "bounds": False, "mesh": True},
                      "enableDefaultSubsources": False},
        ),
    ],
    annotations=[
        AnnotationLayerSpec(
            file_uid="merfish-points",local_path=precomputed_annotations_path, obs_type="point",
            feature_type="gene", label="Transcripts", stroke_width=0.0,
            options={"projectionAnnotationSpacing": 2.4544585683772735,
                     "featureIndexProp": "gene", "pointIndexProp": "gene"},
        ),
    ],
    spatialdata_obs=[
        SpatialDataObsSpec(
            sdata_path=sdata_path,
            obs_feature_matrix_path="tables/gene_expression_baysor/X",  # full path, not "X"
            obs_type="cell", feature_type="gene",

        ),
        SpatialDataObsSpec(
            sdata_path=sdata_path,
            table_path="tables/gene_expression_baysor",
            obs_set_paths=['tables/gene_expression_cellpose/obs/cluster'], obs_set_names=["Cell Types"],
            obs_type="cell",
        ),
    ],

    initial_camera_state={
        "position": [3276962.5, 3271567.5, 1.72], "projectionScale": 11521,
        "projectionOrientation": [-0.0017234950792044401, -0.031710099428892136,
                                   0.02632056176662445, 0.999148964881897],
    },
    show_axis_lines=True,
    layer_per_feature_for_points=True,
    extra_view_types=["featureList", "obsSets"],
)
vc

/Users/tabassumkakar/Development/tissue-map-tools/data/merfish_mouse_ileum/out/merfish_mouse_ileum_precomputed_annotations/molecule_baysor
